# Module 14 Final Report Clinic

**BAN 6003: Data Management and Analytics Integration**

**Not graded.** This clinic replaces the Week 14 lab. Nothing here is submitted or scored.

The Module 14 video explained how to turn model output into a responsible analytics story: business meaning, use boundaries, a clear project story, careful interpretation, ethics, and evidence-based recommendations. This clinic shows the same ideas as short, runnable examples, so you can find and fix the same problems in your own final report.

| Part | What you will see | Suggested time |
|---|---|---|
| 1 | How to use this clinic | 5 minutes |
| 2 | Common report problems and how to fix them (8 short demos) | 40-50 minutes |
| 3 | Worked report audit: a weak report section, the audit, and the revision | 20-30 minutes |
| 4 | Repository organization and a self-check you can copy | 20-30 minutes |
| 5 | Final submission reminders | 5-10 minutes |

# Part 1. How to Use This Clinic

- **It is optional, but strongly recommended.** Most points lost on final projects come from the problems shown here, and all of them are fixable before the deadline.
- **Work in your own project repository.** Keep this clinic open on one side and your final project notebook on the other. After each demo, look at your own report and ask: "Do I have this problem?"
- **Time:** about 1.5-2 hours in total. You can do one part at a time.
- **No answers are required.** Every code cell already runs. Run the notebook from top to bottom (it takes well under a minute), read each demo, then fix your own project.
- **Copy the self-check cell in Part 4** into your project. It checks the required files, the HTML report, large files, `.env`, and the AI Use Disclosure cell.

Each demo in Part 2 follows the same pattern:

> **Problem** (what a weak report does) → **Why it matters** (what the reader or grader loses) → **Fix** (what a strong report does)

The examples use the `nycflights13` data you already know from Modules 12-13, plus a few small made-up tables.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nycflights13 import flights

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score)

pd.set_option("display.max_columns", 20)
RANDOM_STATE = 6003

# One reproducible sample of completed flights, used by several demos below
sample = (flights.dropna(subset=["dep_delay", "arr_delay"])
                 .sample(20_000, random_state=RANDOM_STATE)
                 .copy())
print(f"Full flights table: {len(flights):,} rows. Demo sample: {len(sample):,} completed flights.")

# Part 2. Common Report Problems and How to Fix Them

These eight problems appear in many final reports. Each demo is short on purpose.

## Demo 1. A chart the reader cannot read

**Problem:** The chart has no title, no y-axis label, and no units. The x-axis shows month numbers only.

**Why it matters:** The reader must guess what the chart shows. In the HTML report you are not there to explain it. The rubric asks for labeled tables and charts.

In [ ]:
# WEAK: default plot, no title, no y-axis label, no units
monthly = flights.groupby("month")["arr_delay"].mean()
monthly.plot(figsize=(6, 3))
plt.show()

*Text description:* This chart has no title, no y-axis label, and no units, so a reader cannot tell that it shows average arrival delay in minutes by month.

**Fix:** Add a title that states the message, axis labels with units, readable category names, and a reference line for zero. Then write one sentence below the chart that states the takeaway.

In [ ]:
# FIXED: clear title, axis labels with units, month names, zero line
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(month_names, monthly.values, color="#4C78A8", width=0.7)
ax.axhline(0, color="#444444", linewidth=0.8)
ax.set_title("Average arrival delay by month, NYC departures, 2013")
ax.set_xlabel("Month of departure")
ax.set_ylabel("Average arrival delay (minutes)")
ax.grid(axis="y", color="#DDDDDD", linewidth=0.6)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

*Text description:* Average arrival delay was highest in June and July (about 17 minutes) and December (about 15 minutes), and lowest in September, when flights arrived about 4 minutes early on average.

**Check your report:** Does every chart have a title, axis labels with units, and one sentence that tells the reader what to notice?

## Demo 2. A metric dump with no business meaning

**Problem:** The report prints every number the model produced and moves on.

**Why it matters:** As the video said, *output is not insight*. The reader does not know whether an error of this size is good, bad, or useful for the decision.

In [ ]:
# A simple regression: predict arrival delay (minutes) from departure delay and distance
features = ["dep_delay", "distance"]
X_train, X_test, y_train, y_test = train_test_split(
    sample[features], sample["arr_delay"], test_size=0.25, random_state=RANDOM_STATE)
reg = LinearRegression().fit(X_train, y_train)
pred = reg.predict(X_test)

# WEAK: raw dump, no units, no comparison, no meaning
print(mean_absolute_error(y_test, pred), mean_squared_error(y_test, pred) ** 0.5,
      r2_score(y_test, pred), reg.coef_, reg.intercept_)

**Fix:** Show a small labeled table, compare with a simple baseline, and write one sentence in business language that connects the error size to the decision.

In [ ]:
# FIXED: labeled table with units and a baseline
baseline_pred = np.full(len(y_test), y_train.mean())   # always predict the average delay
mae_model = mean_absolute_error(y_test, pred)
mae_base = mean_absolute_error(y_test, baseline_pred)

metrics_table = pd.DataFrame({
    "Model": ["Baseline (always predict the average)", "Linear regression"],
    "MAE (minutes)": [mae_base, mae_model],
    "RMSE (minutes)": [mean_squared_error(y_test, baseline_pred) ** 0.5,
                       mean_squared_error(y_test, pred) ** 0.5],
}).round(1)
print("Table 1. Test-set error for predicting arrival delay (5,000 test flights)")
display(metrics_table)

print(f"Interpretation: on test flights, predictions are off by about {mae_model:.0f} minutes "
      f"on average, compared with about {mae_base:.0f} minutes for the simple baseline. "
      "This may be accurate enough to flag flights for operational review, "
      "but not for exact gate or crew scheduling.")

**Check your report:** For each metric, can a manager understand the unit, the comparison, and what the number means for the decision?

## Demo 3. Causal language from a regression coefficient

**Problem:** The report reads a coefficient as a cause: "Departure delay **causes** arrival delay, so cutting departure delay by 10 minutes **will** cut arrival delay by 10 minutes."

**Why it matters:** A coefficient from observational data describes an *association* in the historical data, holding the other features in the model constant. It does not prove what will happen if the business changes something. Overclaiming is one of the most common final-report problems, and the rubric checks that you avoid unsupported causal claims.

In [ ]:
coef_dep = reg.coef_[features.index("dep_delay")]

weak = (f"WEAK:  Each extra minute of departure delay causes {coef_dep:.2f} extra minutes of "
        f"arrival delay, so reducing departure delay by 10 minutes will cut arrival delay by "
        f"{10 * coef_dep:.0f} minutes.")
fixed = (f"FIXED: In this 2013 historical data, each extra minute of departure delay is associated "
         f"with about {coef_dep:.2f} extra minutes of predicted arrival delay, holding distance "
         "constant. This is a predictive pattern, not proof of cause; other factors such as weather "
         "and airport congestion may affect both delays.")
print(weak, fixed, sep="\n\n")

**Useful phrases:** *the model suggests*, *is associated with*, *the results are consistent with*, *this may indicate*, *the model can support ... but it cannot prove ...*

**Words to avoid** unless your design supports them: *proves*, *causes*, *will always*, *guarantees*, *risk-free*.

## Demo 4. A model that uses information from the future (leakage)

**Problem:** The team wants to predict late arrivals (15 or more minutes late) **the evening before** each flight, so staff can plan. The model includes `dep_delay`, but departure delay is only known on the day of the flight, after the plane leaves the gate.

**Why it matters:** A feature that is known only *after* the decision time makes the model look much better than it can ever be in real use. This is data leakage. Always state **when** the prediction is made and check that every feature is available at that time.

In [ ]:
target = (sample["arr_delay"] >= 15).astype(int)
sample["day_of_week"] = pd.to_datetime(sample[["year", "month", "day"]]).dt.dayofweek
honest_features = ["hour", "month", "day_of_week", "distance"]   # known the evening before
leaky_features = honest_features + ["dep_delay"]                  # dep_delay is known only later

rows = []
for label, cols in [("Leaky (includes dep_delay)", leaky_features),
                    ("Honest (known the evening before)", honest_features)]:
    Xtr, Xte, ytr, yte = train_test_split(sample[cols], target, test_size=0.25,
                                          random_state=RANDOM_STATE, stratify=target)
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xtr, ytr)
    p = clf.predict(Xte)
    rows.append({"Feature set": label, "Accuracy": accuracy_score(yte, p),
                 "Precision (late)": precision_score(yte, p), "Recall (late)": recall_score(yte, p)})

print(f"Share of late flights in the sample: {target.mean():.0%}")
print("Table 2. Late-arrival classification on 5,000 test flights")
display(pd.DataFrame(rows).round(2))

**Fix:** Report the honest model. Its scores are lower, but they describe what the model can really do at decision time. Then say so in the report, for example:

> "Because the prediction is made the evening before the flight, departure delay was excluded. With schedule-only features, the model catches some late flights but also raises many false alerts, so it is a weak screening tool. Weather forecasts are a possible next feature."

Note: if your decision is made **after** departure (for example, warning connecting passengers), then `dep_delay` is available and is not leakage. The point is to **state the decision time** and check each feature against it. Also check features created from the target (for example, a "late" flag built from `arr_delay`).

## Demo 5. Reporting a merged ABT without row-count and duplicate-key checks

**Problem:** Two tables are merged and the result is used right away. Here, a made-up store lookup table accidentally has the same store twice.

**Why it matters:** A duplicate key on the "one" side of a join silently copies rows. Totals, averages, and model results then change, and the final ABT no longer means "one row per order". The rubric checks row counts, duplicate keys, and missingness before and after integration.

In [ ]:
# Made-up example tables
orders = pd.DataFrame({"order_id": [101, 102, 103, 104, 105, 106],
                       "store_id": ["S1", "S2", "S2", "S3", "S1", "S4"],
                       "amount": [25.0, 40.0, 15.0, 60.0, 30.0, 20.0]})
stores = pd.DataFrame({"store_id": ["S1", "S2", "S2", "S3"],          # S2 appears twice
                       "region": ["North", "South", "South ", "East"]})  # and S4 is missing

# WEAK: merge and move on
abt_weak = orders.merge(stores, on="store_id", how="left")
print(f"Total sales in the weak ABT: ${abt_weak['amount'].sum():.0f}  (the true total is ${orders['amount'].sum():.0f})")

**Fix:** Check the key *before* the merge, use `validate=` so pandas stops you if the relationship is wrong, and compare row counts and missing values *after* the merge. Then report the checks in one short table or sentence.

In [ ]:
# FIXED: check keys, clean the lookup table, validate the merge, then compare row counts
print("Duplicate store_id rows in stores:", stores["store_id"].duplicated().sum())

stores_clean = (stores.assign(region=stores["region"].str.strip())
                      .drop_duplicates(subset="store_id"))
abt = orders.merge(stores_clean, on="store_id", how="left", validate="many_to_one")

checks = pd.DataFrame({
    "Check": ["Rows before merge", "Rows after merge", "Duplicate order_id after merge",
              "Orders with no matching store"],
    "Result": [len(orders), len(abt), abt["order_id"].duplicated().sum(), abt["region"].isna().sum()],
})
print("Table 3. Integration checks (one row per order expected)")
display(checks)

In the report, explain the result in one sentence: *"The ABT keeps one row per order (6 rows before and after the merge). One duplicate store record was removed, and one order (store S4) has no region and is shown as missing."*

## Demo 6. Absolute file paths and hidden notebook state

**Problem 1 - absolute paths:** `pd.read_csv("C:/Users/maria/Downloads/orders.csv")` works only on Maria's laptop. On the instructor's computer, or in Codespaces, the notebook fails on the first cell.

**Problem 2 - hidden state:** A notebook can look fine on your screen but fail after **Restart & Run All**, because a cell uses a variable that was created in a cell you later deleted or edited.

**Why it matters:** The rubric asks for a reproducible notebook, and the HTML report must be exported after Restart & Run All. If your notebook fails from a fresh start, your report shows errors or old results.

**Fix 1:** Use paths relative to the repository root, built with `pathlib`.

In [ ]:
from pathlib import Path

# FIXED: find the repository root whether the notebook runs from the root or from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Example usage in your project (not run here):
#   orders = pd.read_csv(DATA_DIR / "orders.csv")
#   abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
print("Relative path example:", (DATA_DIR / "orders.csv").relative_to(PROJECT_ROOT))

**Fix 2:** Restart the kernel and run all cells often, not only at the end. The cell below *simulates* a fresh restart: it runs the saved cells in order in an empty workspace and shows which cell breaks.

In [ ]:
# The cells as they are saved in a notebook. The cell that created clean_df was deleted.
saved_cells = [
    "import pandas as pd",
    "summary = clean_df.groupby('carrier').size()",
    "print(summary.head())",
]
fresh_workspace = {}  # like a new kernel: no variables yet
for number, cell_code in enumerate(saved_cells, start=1):
    try:
        exec(cell_code, fresh_workspace)
        print(f"Cell {number}: ran")
    except NameError as error:
        print(f"Cell {number}: FAILS after Restart & Run All -> {error}")
        break

## Demo 7. Debug output and huge printed tables

**Problem:** The report is full of `print("here")` lines, long column lists, full `dtypes` output, and whole tables. The important results are lost.

**Why it matters:** The reader must scroll through output that you never explain. The final page asks you to remove unnecessary debugging output, and the rubric rewards a readable report.

In [ ]:
# WEAK: debugging output left in the report (shortened here)
print("DEBUG got here")
print(sample.shape)
print(list(sample.columns))
print("check", sample["arr_delay"].isna().sum(), sample["dep_delay"].isna().sum(), sample["tailnum"].isna().sum())
print(sample.dtypes.to_dict())

**Fix:** Show one labeled summary table and a small preview with `head()`. Keep long checks in the appendix, or delete them once they pass.

In [ ]:
# FIXED: one labeled summary and a short preview
key_cols = ["dep_delay", "arr_delay", "distance"]
summary = sample[key_cols].describe().T[["count", "mean", "50%", "min", "max"]].round(1)
summary.columns = ["Rows", "Mean", "Median", "Min", "Max"]
print("Table 4. Key numeric columns in the demo sample (delays in minutes, distance in miles)")
display(summary)

print("Table 5. First 5 rows of selected columns")
display(sample[["month", "day", "carrier", "origin", "dest", "dep_delay", "arr_delay"]].head())

## Demo 8. Missing limitations, ethics, and use boundaries

**Problem:** The report ends with: *"Limitations: none. The model works well and should be used for all flights."*

**Why it matters:** Every model has limits. A report without them looks overconfident, and the reader does not know when the result should **not** be used. The rubric requires limitations, ethics, and responsible use. If you did not check fairness, you should not claim that the model is fair.

**Fix:** One short paragraph and a small "use boundaries" table are often enough. A quick group check (here: errors by origin airport) shows whether the model works about equally well for different groups.

In [ ]:
# A quick group check: is the regression error similar across origin airports?
test_view = sample.loc[X_test.index, ["origin"]].assign(abs_error=np.abs(y_test - pred))
by_origin = (test_view.groupby("origin")["abs_error"]
             .agg(["count", "mean"]).round(1)
             .rename(columns={"count": "Test flights", "mean": "MAE (minutes)"}))
print("Table 6. Regression error by origin airport")
display(by_origin)

use_boundaries = pd.DataFrame({
    "The model CAN support": ["Flagging flights for review", "Comparing typical delay patterns",
                              "Planning further analysis"],
    "The model should NOT be used for": ["Exact gate or crew schedules", "Claims about what causes delay",
                                         "Automatic decisions without human review"],
})
print("Table 7. Use boundaries")
display(use_boundaries)

**Example limitations paragraph (adapt it to your project):**

> *Limitations and responsible use.* The model was built on 2013 flights from three New York airports, so it may not reflect newer schedules or other airports. Rows with missing delay values (mostly cancelled flights) were removed, so the model describes completed flights only. The features are simple, and weather was not included. Results show associations, not causes. Error was checked by origin airport (Table 6), but not by carrier or route, so fairness across those groups is unknown. The model should support human review of likely delays, not automatic decisions that affect passengers or staff.

**Check your report:** Does it say what the result can and cannot be used for, what you checked, and what remains unknown?

# Part 3. Worked Report Audit

This part shows the full process on one example: a weak report section, an audit against the final project rubric categories, and a revised version.

> **This example is made up.** "Riverwalk Grocery" is a fictional online grocery company. The numbers are invented for teaching only.

## 3.1 The weak excerpt

---

**Executive summary.** I built a decision tree to predict late deliveries for Riverwalk Grocery. The model has 95% accuracy, which proves it works very well. Distance causes late deliveries.

**Results.** Accuracy = 0.95. Precision = 0.78. Recall = 0.62. Depth = 5. Features: distance_km, items, hour, driver_id, delivered_at_hour.

**Recommendation.** Riverwalk should use the model for all orders and give a warning to drivers whose orders are predicted to be late.

---

## 3.2 The audit

Read the excerpt as a grader would, one rubric category at a time.

| Final rubric category (points) | What the excerpt shows | What is missing or wrong | How to fix it |
|---|---|---|---|
| 1. Business framing and project setup (8) | "Predict late deliveries" | No business problem, no user, no decision, no reason why it matters | Name the user (dispatch team), the decision (which orders get extra attention), and why late orders matter (refunds, complaints) |
| 2. Data understanding, cleaning, transformation, and validation (25) | Nothing | No data source, no profiling, no cleaning decisions, no definition of "late" | Summarize the source, row counts, key cleaning steps, and how the target was built (for example, "late = delivered more than 30 minutes after the promised window") |
| 3. Integration, database, SQL, and ABT preparation (25) | Nothing | No unit of analysis, no keys, no merge checks, no data dictionary | State "one row per order", the tables joined and keys used, row counts before and after, and point to the data dictionary file |
| 4. Modeling and analytics application (20) | Metrics and a feature list | Accuracy is never compared with a baseline (91% of orders are on time, so "always on time" is already 91% accurate); `delivered_at_hour` is known only after delivery, so the scores are inflated by leakage; "proves" and "causes" are unsupported | Remove the leaky feature, add a baseline, explain precision and recall for late orders, use cautious language |
| 5. Interpretation, limitations, ethics, and recommendations (12) | One recommendation | No business meaning of the metrics, no limitations, no ethics; warning drivers based on a model with many false alerts is unfair to them, and `driver_id` lets the model learn about individual people | Interpret metrics in business terms, add limitations, remove `driver_id`, state use boundaries, make the recommendation specific and cautious with a next step |
| 6. Reproducibility, submission package, and communication (10) | Short text | No tables or charts are referenced, no labels, no link between results and earlier sections | Use a labeled results table, refer to it in the text, and keep the executive summary to the full workflow in 4-6 sentences |

**The biggest problems:** an accuracy number with no baseline that is inflated by a leaky feature, causal language, and a recommendation that could harm drivers. Notice that the data management work (Categories 2 and 3, half of the points) is invisible in the weak excerpt, even if the student actually did it in the notebook.

## 3.3 The revised excerpt

---

**Executive summary.** Riverwalk Grocery's dispatch team wants to know which online orders are likely to be delivered late, so they can add a driver or contact the customer early. I combined order, store, and delivery tables into an ABT with one row per order (48,210 orders, January-June), after removing 312 duplicate order records and 1,450 orders with no delivery address (3% of orders). A decision tree using only information available when the order is placed was compared with a simple "always on time" baseline. The model finds about half of the late orders, but about 7 of every 10 orders it flags are actually on time. It may support a daily review list, not automatic actions, and it should be tested on newer data before wider use.

**Results.** Table 8 compares the model with the baseline on 12,050 test orders. The baseline has higher accuracy (0.91 against 0.84) only because 91% of orders are on time and it never flags any order, so it finds no late orders at all. For late orders, the model's recall is 0.55 (it finds 55% of late orders) and its precision is 0.30 (30% of flagged orders are really late). The earlier version reported 0.95 accuracy, but it used `delivered_at_hour`, which is known only after delivery. Orders over 12 km and orders placed between 4 pm and 7 pm were associated with a higher predicted chance of being late. This is a predictive pattern, not proof that distance causes delays.

| Table 8. Test-set results (12,050 orders) | Accuracy | Precision (late) | Recall (late) |
|---|---:|---:|---:|
| Baseline: predict "on time" for every order | 0.91 | not defined | 0.00 |
| Decision tree (depth 5, order-time features only) | 0.84 | 0.30 | 0.55 |

**Limitations and responsible use.** The data covers six months and has no weather or traffic data. Orders without an address were removed, so results describe complete orders only. `driver_id` was removed from the features so the model does not score individual people, and `delivered_at_hour` was removed because it is known only after delivery (leakage). Errors were not compared across neighborhoods, so it is unknown whether some areas receive more false alerts.

**Recommendations.**
1. Use the model to build a daily review list of orders with a high predicted chance of being late, and let a dispatcher decide what to do. Do not use it to rate or warn individual drivers.
2. Before wider use, test the model on July-September orders and compare its errors across neighborhoods.
3. Add weather and traffic data in the next version, and check whether recall for late orders improves.

---

**What changed:** the summary now covers the full workflow (problem, data preparation, ABT, method, main finding, limitation, recommendation); the metrics have a baseline and a business meaning; leakage and causal language are gone; the ethics issue is handled; and each recommendation is specific, cautious, and has a next validation step.

# Part 4. Repository Organization

Your instructor opens your repository before your report. A clean repository makes your work easy to find and easy to run. The rubric gives points for an organized repository, a complete package, and reproducible code.

## 4.1 Expected layout

```text
your-project-repository/
├── README.md                 project title, question, how to run, file list
├── requirements.txt          packages your notebook needs
├── .gitignore                files git should NOT track (.env, checkpoints, large raw files)
├── data/                     raw data, or a README that explains where to get it
├── notebooks/
│   └── <your_notebook>.ipynb final report notebook (one final notebook)
├── outputs/
│   ├── <option>_final_abt.csv
│   └── <option>_data_dictionary.csv
├── reports/
│   ├── <your_notebook>.html  exported after Restart & Run All
│   └── slides.pdf            optional, only if you present
└── sql/                      optional: SQL files, if your project uses them
```

## 4.2 What does not belong in your repository

| Do not commit | Why | What to do instead |
|---|---|---|
| Very large raw files (over 50 MB) | GitHub warns at 50 MB and rejects files over 100 MB | Add them to `.gitignore` and explain in README.md where to download them |
| `.env` files, passwords, API keys, database passwords | Anyone who opens a public repository can read them | Keep `.env` in `.gitignore` (the template already does this) |
| Presentation videos | Too large for GitHub; the optional presentation is submitted in a separate Canvas assignment | Upload to Canvas or share a link there |
| Scratch notebooks (`test.ipynb`, `Untitled1.ipynb`, `final_v2_REAL.ipynb`) | The grader cannot tell which notebook is the final one | Delete them or move them out of the repository; keep one final notebook in `notebooks/` |
| `.ipynb_checkpoints/`, `__pycache__/`, `.DS_Store` | Automatic system files | Already listed in the template `.gitignore` |

## 4.3 A README template you can copy

Copy this into your project's `README.md` and replace the text in angle brackets.

```markdown
# <Project title>

BAN 6003: Data Management and Analytics Integration - Final Project
<Your name> - <Semester>

## Business question
<One or two sentences: the problem, who uses the result, and the analytical question.>

## Data
<Data source names and links. If a raw file is too large for GitHub, say where to download it and where to save it (for example, data/orders.csv).>

## How to run
1. Install packages: python -m pip install -r requirements.txt
2. Open notebooks/<your_notebook>.ipynb and choose Restart & Run All.
3. The notebook saves outputs/<option>_final_abt.csv and outputs/<option>_data_dictionary.csv.

## Files
- notebooks/<your_notebook>.ipynb - final report notebook
- reports/<your_notebook>.html - HTML report exported after Restart & Run All
- outputs/<option>_final_abt.csv - final ABT (one row per <unit of analysis>)
- outputs/<option>_data_dictionary.csv - one row per ABT column
```

## 4.4 File naming tips

- Use lowercase letters, numbers, and underscores: `olist_late_delivery_report.ipynb`, not `My Final Project (2) FINAL.ipynb`.
- Spaces and brackets in names make terminal commands such as `jupyter nbconvert` harder to type.
- The HTML report must have the same base name as the notebook: `notebooks/olist_late_delivery_report.ipynb` → `reports/olist_late_delivery_report.html`. The export command does this for you.
- Use the exact required output names: `outputs/<option>_final_abt.csv` and `outputs/<option>_data_dictionary.csv`, with the same `<option>` in both.

## 4.5 Check git status and write clear commit messages

Your commit history is reviewed, so commit and push after each major step, not all at once at the end. In a terminal at the repository root:

```bash
git status                 # what changed, and what is not tracked yet
git ls-files               # every file that git tracks (check: no .env, no videos)
git log --oneline -10      # your last 10 commits

git add notebooks/olist_late_delivery_report.ipynb outputs/
git commit -m "Add duplicate-key checks before order-store merge"
git push
```

Good commit messages say **what changed**: "Clean review dates and remove duplicate orders", "Save final ABT and data dictionary", "Export HTML report after Restart & Run All". Weak messages: "update", "asdf", "final", "final2".

## 4.6 The self-check cell

The next cell defines `final_report_self_check()`. It reads files only; it does not change or delete anything. It checks:

1. `notebooks/` has an `.ipynb` file (if there are several, it checks the newest one and lists the others);
2. `reports/` has an `.html` file with the same name that is **newer** than the notebook;
3. `outputs/*_final_abt.csv` and `outputs/*_data_dictionary.csv` exist;
4. no file is larger than 50 MB (files ignored by `.gitignore` are noted, not flagged);
5. no `.env` file is tracked by git (skipped with a message if git is not available);
6. the notebook has a markdown cell that starts with "AI Use Disclosure".

**How to use it in your project:**

- Copy the whole next cell into a new cell at the end of your project notebook and run it. It finds your repository root by itself, whether the notebook runs from the root or from `notebooks/`.
- Fix the items marked `FIX`. The HTML item passes only after your final export, so a `FIX` there is normal before you export.
- **Delete the self-check cell before your final Restart & Run All and export**, so its output does not appear in your report. You can also paste the cell into a Python session in a terminal opened at the repository root (type `python`, paste, press Enter twice, then type `final_report_self_check()`).
- If you just cloned or opened a new Codespace, file times may be reset. If the HTML item fails, simply export again.

In [ ]:
# ===== BAN 6003 final report self-check (copy this whole cell) =====
# Run it from your project notebook or from the repository root.
# It only reads files. It does not change or delete anything.
import json
import os
import shutil
import subprocess
from pathlib import Path

SIZE_LIMIT_MB = 50


def find_repo_root(start="."):
    """Return the project folder: the first folder (going up) that has notebooks/ or .git."""
    here = Path(start).resolve()
    for folder in [here, *here.parents]:
        if (folder / "notebooks").is_dir() or (folder / ".git").exists():
            return folder
    return here


def _git(root, *args):
    """Run a git command in root. Return its output, or None if git is not available."""
    if shutil.which("git") is None:
        return None
    try:
        result = subprocess.run(["git", *args], cwd=root, capture_output=True,
                                text=True, timeout=30)
    except (OSError, subprocess.SubprocessError):
        return None
    return result.stdout if result.returncode == 0 else None


def _starts_with_ai_disclosure(text):
    first_line = text.strip().splitlines()[0] if text.strip() else ""
    cleaned = first_line.lstrip("#*>_ \t").lower()
    return cleaned.startswith("ai use disclosure")


def final_report_self_check(repo_root=None, notebook_name=None):
    root = find_repo_root() if repo_root is None else Path(repo_root).resolve()
    rows = []  # (status, message, how to fix)

    def add(status, message, fix=""):
        rows.append((status, message, fix))

    # 1. Final notebook in notebooks/
    nb_dir = root / "notebooks"
    notebooks = sorted(nb_dir.glob("*.ipynb")) if nb_dir.is_dir() else []
    notebook = None
    if notebook_name:
        candidate = nb_dir / notebook_name
        if candidate.exists():
            notebook = candidate
    elif notebooks:
        notebook = max(notebooks, key=lambda p: p.stat().st_mtime)
    if notebook is None:
        add("FIX", "No final notebook found in notebooks/.",
            "Save your final report notebook as notebooks/<your_notebook>.ipynb.")
    else:
        add("OK", f"Final notebook found: notebooks/{notebook.name}")
        if len(notebooks) > 1 and not notebook_name:
            others = ", ".join(p.name for p in notebooks if p != notebook)
            add("NOTE", f"More than one notebook in notebooks/. Checked the newest one. Others: {others}",
                "Move scratch notebooks out of the repository, or call "
                "final_report_self_check(notebook_name='your_notebook.ipynb').")

    # 2. HTML report in reports/, newer than the notebook
    if notebook is not None:
        html = root / "reports" / (notebook.stem + ".html")
        if not html.exists():
            other_html = sorted((root / "reports").glob("*.html")) if (root / "reports").is_dir() else []
            extra = f" (found: {', '.join(p.name for p in other_html)})" if other_html else ""
            add("FIX", f"No HTML report named reports/{html.name}{extra}.",
                "Restart & Run All, save, then from the repository root run: "
                f"jupyter nbconvert --to html --output-dir reports notebooks/{notebook.name}")
        elif html.stat().st_mtime + 1 < notebook.stat().st_mtime:
            add("FIX", f"reports/{html.name} is older than the notebook, so it may show old results.",
                "Restart & Run All, save, and export the HTML again as the very last step.")
        else:
            add("OK", f"HTML report is up to date: reports/{html.name}")

    # 3. Required outputs files
    for pattern, label in [("*_final_abt.csv", "Final ABT"),
                           ("*_data_dictionary.csv", "Data dictionary")]:
        found = sorted((root / "outputs").glob(pattern)) if (root / "outputs").is_dir() else []
        if found:
            add("OK", f"{label} found: outputs/{found[0].name}")
        else:
            add("FIX", f"{label} not found: no file matches outputs/{pattern}.",
                f"Save it from your notebook with code, for example outputs/olist{pattern[1:]}.")

    # 4. Large files (GitHub rejects files over 100 MB)
    tracked_text = _git(root, "ls-files")
    tracked = set(tracked_text.splitlines()) if tracked_text is not None else None
    large = []
    for folder, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d != ".git"]
        for name in files:
            path = Path(folder) / name
            try:
                size_mb = path.stat().st_size / 1_000_000
            except OSError:
                continue
            if size_mb > SIZE_LIMIT_MB:
                large.append((path.relative_to(root).as_posix(), size_mb))
    if not large:
        add("OK", f"No file larger than {SIZE_LIMIT_MB} MB.")
    for rel, size_mb in large:
        if tracked is not None and rel not in tracked and _git(root, "check-ignore", rel) is not None:
            add("NOTE", f"{rel} is {size_mb:.0f} MB but is ignored by .gitignore, so it stays out of GitHub.")
        else:
            add("FIX", f"{rel} is {size_mb:.0f} MB (over {SIZE_LIMIT_MB} MB; GitHub rejects files over 100 MB).",
                "Remove it from the repository, add it to .gitignore, and explain in README.md "
                "where the data comes from.")

    # 5. No .env file tracked by git
    if tracked is None:
        add("SKIP", "Could not run git here, so the .env check was skipped.",
            "In a terminal at the repository root, run: git ls-files | grep .env")
    else:
        env_files = [p for p in tracked if Path(p).name == ".env"]
        if env_files:
            add("FIX", f"A .env file is tracked by git: {', '.join(env_files)}",
                "Run: git rm --cached .env   then add .env to .gitignore and commit.")
        else:
            add("OK", "No .env file is tracked by git.")

    # 6. AI Use Disclosure markdown cell
    if notebook is not None:
        try:
            cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
        except (OSError, ValueError):
            cells = []
            add("FIX", f"Could not read notebooks/{notebook.name} as a notebook file.")
        md_cells = ["".join(c.get("source", "")) for c in cells if c.get("cell_type") == "markdown"]
        hits = [i for i, text in enumerate(md_cells) if _starts_with_ai_disclosure(text)]
        if not hits:
            add("FIX", "No markdown cell starting with 'AI Use Disclosure' was found.",
                "Add a markdown cell at the end: 'AI Use Disclosure: ...' (write 'None' if you used no AI tools).")
        elif hits[-1] != len(md_cells) - 1:
            add("NOTE", "AI Use Disclosure cell found, but it is not the last markdown cell.",
                "Move it to the end of the notebook.")
        else:
            add("OK", "AI Use Disclosure cell found at the end of the notebook.")

    # Print a friendly checklist
    print(f"Final report self-check for: {root}")
    print("-" * 60)
    for status, message, fix in rows:
        print(f"[{status:<4}] {message}")
        if fix and status != "OK":
            print(f"        How to fix: {fix}")
    n_fix = sum(1 for r in rows if r[0] == "FIX")
    print("-" * 60)
    if n_fix == 0:
        print("All required checks passed. Still read your HTML report once from top to bottom.")
    else:
        print(f"{n_fix} item(s) to fix before you submit.")
    return rows

### Demo: a passing example

The next cell creates a tiny **demo project** in a temporary folder (outside your repository), with every required file, and runs the self-check on it. This shows what a passing result looks like. The temporary folder is removed automatically when the cell finishes.

In [ ]:
import tempfile
import time
import nbformat

with tempfile.TemporaryDirectory() as tmp:
    demo = Path(tmp) / "demo_project"
    for folder in ["data", "notebooks", "outputs", "reports"]:
        (demo / folder).mkdir(parents=True)
    (demo / "README.md").write_text("# Demo project\n")
    (demo / ".gitignore").write_text(".env\n.ipynb_checkpoints/\n")

    demo_nb = nbformat.v4.new_notebook()
    demo_nb.cells = [nbformat.v4.new_markdown_cell("# Demo final report"),
                     nbformat.v4.new_code_cell("print('analysis here')"),
                     nbformat.v4.new_markdown_cell("## AI Use Disclosure\n\nNone.")]
    nbformat.write(demo_nb, demo / "notebooks" / "demo_report.ipynb")

    pd.DataFrame({"order_id": [1, 2], "late": [0, 1]}).to_csv(
        demo / "outputs" / "demo_final_abt.csv", index=False)
    pd.DataFrame({"column_name": ["order_id", "late"], "data_type": ["int", "int"],
                  "description": ["Order key", "1 if delivered late"], "source": ["orders", "derived"],
                  "transformation": ["none", "delivered_at > promised_at"], "notes": ["", ""]}).to_csv(
        demo / "outputs" / "demo_data_dictionary.csv", index=False)

    html = demo / "reports" / "demo_report.html"
    html.write_text("<html><body>Demo report</body></html>")
    later = time.time() + 5
    os.utime(html, (later, later))   # the HTML was exported after the notebook was saved

    if shutil.which("git"):
        subprocess.run(["git", "init", "-q"], cwd=demo, capture_output=True)
        subprocess.run(["git", "add", "."], cwd=demo, capture_output=True)

    final_report_self_check(demo)

### Demo: running it in this clinic repository

This clinic repository is not a final project, so the check below should report items to fix (no HTML report, no `outputs/` files, no AI Use Disclosure cell). This is what you will see in your own project if something is missing. It is expected here and is not an error.

In [ ]:
checklist = final_report_self_check()

# Part 5. Final Reminders

The final submission is due right after Module 15. These items match the Canvas page "BAN 6003 Final Project Submission".

## Required in your GitHub repository

- [ ] Final project notebook: `notebooks/<your_notebook>.ipynb`
- [ ] HTML report: `reports/<your_notebook>.html`, exported **after Restart & Run All** from the repository root with:
  `jupyter nbconvert --to html --output-dir reports notebooks/<your_notebook>.ipynb`
- [ ] Final ABT: `outputs/<option>_final_abt.csv`, created by code in your notebook, and the same table your model or analysis uses
- [ ] Data dictionary: `outputs/<option>_data_dictionary.csv`, one row per ABT column, with at least: column name, data type, description, source, transformation, notes
- [ ] Supporting files needed to reproduce your work (for example, `requirements.txt` or SQL files)
- [ ] An "AI Use Disclosure" markdown cell at the end of the notebook (write "None." if you used no AI tools)

## Before you export

- [ ] Restart the kernel and run all cells; no error messages
- [ ] Unnecessary debugging output removed (Demo 7)
- [ ] Markdown explanations complete; no starter-code placeholders left
- [ ] Tables and charts labeled, each with a one-sentence takeaway (Demo 1)
- [ ] Notebook saved, then exported to HTML

## Report content

- [ ] The 12-section report structure from the Final page (title and executive summary through recommendations, plus an optional appendix)
- [ ] Data management work is visible: profiling, cleaning, integration checks, ABT unit of analysis, data dictionary (Demo 5)
- [ ] Metrics interpreted in business language, compared with a baseline (Demo 2)
- [ ] Cautious, non-causal language (Demo 3) and no leakage (Demo 4)
- [ ] Limitations, ethics, and responsible use (Demo 8)
- [ ] 2-3 specific recommendations, each connected to evidence and limitations, with a next step (Part 3)

## Submit

- [ ] Commit regularly; your commit history is reviewed
- [ ] Push the latest version, then submit the repository link in Canvas
- [ ] Public repository, or invite the instructor if it is private
- [ ] No video files in GitHub. The recorded presentation is **optional** (up to 2 bonus points, no penalty if skipped) and is submitted in the separate Canvas assignment "Final Presentation Recording (Optional Bonus)". Slides (PDF or PPTX) may go in `reports/`.

You remain responsible for all work you submit, and you should be able to explain any part of your code or analysis in your own words.